# 08 Compare Models

This notebook compares different machine learning models for predicting NFL game outcomes.

The previous best model used logistic regression with expanded training data from 2018 through 2024 and tested on the 2025 season.

Previous best accuracy: 62.46%

This notebook compares:
- Logistic Regression
- Random Forest
- Gradient Boosting

In [1]:
# Import packages
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report

In [2]:
# Load the best modeling dataset so far
modeling_data = pd.read_csv("../data/processed/modeling_dataset_expanded_2018_2025.csv")

modeling_data.head()

,season,week,game_id,gameday,home_team,away_team,home_score,away_score,home_team_won,home_point_diff,...,away_last3_avg_point_diff,away_last3_win_pct,avg_points_scored_diff,avg_points_allowed_diff,avg_point_diff_diff,win_pct_diff,last3_avg_points_scored_diff,last3_avg_points_allowed_diff,last3_avg_point_diff_diff,last3_win_pct_diff
0,2018,1,2018_01_ATL_PHI,2018-09-06,PHI,ATL,18.0,12.0,1,6.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2018,1,2018_01_BUF_BAL,2018-09-09,BAL,BUF,47.0,3.0,1,44.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2018,1,2018_01_PIT_CLE,2018-09-09,CLE,PIT,21.0,21.0,0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2018,1,2018_01_CIN_IND,2018-09-09,IND,CIN,23.0,34.0,0,-11.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2018,1,2018_01_TEN_MIA,2018-09-09,MIA,TEN,27.0,20.0,1,7.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [3]:
# Check data
modeling_data.shape
modeling_data["season"].value_counts().sort_index()
modeling_data.columns.tolist()

['season',
 'week',
 'game_id',
 'gameday',
 'home_team',
 'away_team',
 'home_score',
 'away_score',
 'home_team_won',
 'home_point_diff',
 'home_games_played_before',
 'home_avg_points_scored_before',
 'home_avg_points_allowed_before',
 'home_avg_point_diff_before',
 'home_win_pct_before',
 'home_last3_avg_points_scored',
 'home_last3_avg_points_allowed',
 'home_last3_avg_point_diff',
 'home_last3_win_pct',
 'away_games_played_before',
 'away_avg_points_scored_before',
 'away_avg_points_allowed_before',
 'away_avg_point_diff_before',
 'away_win_pct_before',
 'away_last3_avg_points_scored',
 'away_last3_avg_points_allowed',
 'away_last3_avg_point_diff',
 'away_last3_win_pct',
 'avg_points_scored_diff',
 'avg_points_allowed_diff',
 'avg_point_diff_diff',
 'win_pct_diff',
 'last3_avg_points_scored_diff',
 'last3_avg_points_allowed_diff',
 'last3_avg_point_diff_diff',
 'last3_win_pct_diff']

In [4]:
# Choose features and target
features = [
    "avg_points_scored_diff",
    "avg_points_allowed_diff",
    "avg_point_diff_diff",
    "win_pct_diff",
    "last3_avg_points_scored_diff",
    "last3_avg_points_allowed_diff",
    "last3_avg_point_diff_diff",
    "last3_win_pct_diff"
]

target = "home_team_won"

In [5]:
# Create train and test data
train_data = modeling_data[
    modeling_data["season"].between(2018, 2024)
].copy()

test_data = modeling_data[
    modeling_data["season"] == 2025
].copy()

X_train = train_data[features]
y_train = train_data[target]

X_test = test_data[features]
y_test = test_data[target]

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])

Training rows: 1942
Testing rows: 285


In [6]:
# Check missing values
print("Missing values in X_train:")
print(X_train.isna().sum())

print("\nMissing values in X_test:")
print(X_test.isna().sum())

Missing values in X_train:
avg_points_scored_diff           0
avg_points_allowed_diff          0
avg_point_diff_diff              0
win_pct_diff                     0
last3_avg_points_scored_diff     0
last3_avg_points_allowed_diff    0
last3_avg_point_diff_diff        0
last3_win_pct_diff               0
dtype: int64

Missing values in X_test:
avg_points_scored_diff           0
avg_points_allowed_diff          0
avg_point_diff_diff              0
win_pct_diff                     0
last3_avg_points_scored_diff     0
last3_avg_points_allowed_diff    0
last3_avg_point_diff_diff        0
last3_win_pct_diff               0
dtype: int64


In [7]:
# Create the models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        max_depth=5
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        random_state=42
    )
}

In [8]:
# Train and evaluate each model
model_results = []

for model_name, model in models.items():
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    
    accuracy = accuracy_score(y_test, y_pred)
    
    model_results.append({
        "model": model_name,
        "accuracy": accuracy
    })

model_results_df = pd.DataFrame(model_results)

model_results_df

,model,accuracy
0,Logistic Regression,0.624561
1,Random Forest,0.607018
2,Gradient Boosting,0.564912


In [9]:
# Format accuracy as percent
model_results_df["accuracy_percent"] = (
    model_results_df["accuracy"] * 100
).round(2)

model_results_df = model_results_df.sort_values(
    "accuracy",
    ascending=False
)

model_results_df

,model,accuracy,accuracy_percent
0,Logistic Regression,0.624561,62.46
1,Random Forest,0.607018,60.70
2,Gradient Boosting,0.564912,56.49


In [10]:
# Compare to previous best
previous_best = 0.6246

model_results_df["difference_from_previous_best"] = (
    model_results_df["accuracy"] - previous_best
)

model_results_df["difference_percent_points"] = (
    model_results_df["difference_from_previous_best"] * 100
).round(2)

model_results_df

,model,accuracy,accuracy_percent,difference_from_previous_best,difference_percent_points
0,Logistic Regression,0.624561,62.46,-0.000039,-0.00
1,Random Forest,0.607018,60.70,-0.017582,-1.76
2,Gradient Boosting,0.564912,56.49,-0.059688,-5.97


In [11]:
# Pick the best model
best_model_name = model_results_df.iloc[0]["model"]
best_accuracy = model_results_df.iloc[0]["accuracy"]

print("Best model:", best_model_name)
print("Best accuracy:", f"{best_accuracy:.2%}")

Best model: Logistic Regression
Best accuracy: 62.46%


In [12]:
# Re-train best model for detailed results
best_model = models[best_model_name]

best_model.fit(X_train, y_train)

best_pred = best_model.predict(X_test)
best_prob = best_model.predict_proba(X_test)[:, 1]

In [13]:
# Classification report for best model
print(classification_report(y_test, best_pred))

              precision    recall  f1-score   support

           0       0.64      0.46      0.53       133
           1       0.62      0.77      0.69       152

    accuracy                           0.62       285
   macro avg       0.63      0.61      0.61       285
weighted avg       0.63      0.62      0.61       285



In [14]:
# Create best model predictions results
results = test_data[
    [
        "season",
        "week",
        "game_id",
        "gameday",
        "home_team",
        "away_team",
        "home_score",
        "away_score",
        "home_team_won"
    ]
].copy()

results["predicted_home_win"] = best_pred
results["home_win_probability"] = best_prob

results["predicted_winner"] = results.apply(
    lambda row: row["home_team"] if row["predicted_home_win"] == 1 else row["away_team"],
    axis=1
)

results["actual_winner"] = results.apply(
    lambda row: row["home_team"] if row["home_team_won"] == 1 else row["away_team"],
    axis=1
)

results["correct_prediction"] = (
    results["predicted_winner"] == results["actual_winner"]
)

results.head(10)

,season,week,game_id,gameday,home_team,away_team,home_score,away_score,home_team_won,predicted_home_win,home_win_probability,predicted_winner,actual_winner,correct_prediction
1942,2025,1,2025_01_DAL_PHI,2025-09-04,PHI,DAL,24.0,20.0,1,1,0.547025,PHI,PHI,True
1943,2025,1,2025_01_KC_LAC,2025-09-05,LAC,KC,27.0,21.0,1,1,0.547025,LAC,LAC,True
1944,2025,1,2025_01_TB_ATL,2025-09-07,ATL,TB,20.0,23.0,0,1,0.547025,ATL,TB,False
1945,2025,1,2025_01_CIN_CLE,2025-09-07,CLE,CIN,16.0,17.0,0,1,0.547025,CLE,CIN,False
1946,2025,1,2025_01_MIA_IND,2025-09-07,IND,MIA,33.0,8.0,1,1,0.547025,IND,IND,True
1947,2025,1,2025_01_CAR_JAX,2025-09-07,JAX,CAR,26.0,10.0,1,1,0.547025,JAX,JAX,True
1948,2025,1,2025_01_LV_NE,2025-09-07,NE,LV,13.0,20.0,0,1,0.547025,NE,LV,False
1949,2025,1,2025_01_ARI_NO,2025-09-07,NO,ARI,13.0,20.0,0,1,0.547025,NO,ARI,False
1950,2025,1,2025_01_PIT_NYJ,2025-09-07,NYJ,PIT,32.0,34.0,0,1,0.547025,NYJ,PIT,False
1951,2025,1,2025_01_NYG_WAS,2025-09-07,WAS,NYG,21.0,6.0,1,1,0.547025,WAS,WAS,True


In [15]:
# Accuracy by week
accuracy_by_week = (
    results.groupby("week")["correct_prediction"]
    .mean()
    .reset_index()
)

accuracy_by_week["accuracy_percent"] = (
    accuracy_by_week["correct_prediction"] * 100
).round(2)

accuracy_by_week = accuracy_by_week.drop(columns=["correct_prediction"])

accuracy_by_week

,week,accuracy_percent
0,1,56.25
1,2,75.00
2,3,56.25
3,4,50.00
4,5,42.86
5,6,46.67
6,7,73.33
7,8,61.54
8,9,57.14
9,10,71.43


In [16]:
# Accuracy by confidence level
def get_confidence(prob):
    if prob >= 0.65 or prob <= 0.35:
        return "High"
    elif prob >= 0.57 or prob <= 0.43:
        return "Medium"
    else:
        return "Low"

results["confidence"] = results["home_win_probability"].apply(get_confidence)

confidence_accuracy = (
    results.groupby("confidence")["correct_prediction"]
    .agg(["count", "mean"])
    .reset_index()
)

confidence_accuracy["accuracy_percent"] = (
    confidence_accuracy["mean"] * 100
).round(2)

confidence_accuracy = confidence_accuracy.rename(
    columns={"count": "number_of_games"}
)

confidence_accuracy = confidence_accuracy.drop(columns=["mean"])

confidence_accuracy

,confidence,number_of_games,accuracy_percent
0,High,81,72.84
1,Low,123,58.54
2,Medium,81,58.02


In [17]:
# Feature importance for tree-based models
if hasattr(best_model, "feature_importances_"):
    feature_importance = pd.DataFrame({
        "feature": features,
        "importance": best_model.feature_importances_
    })

    feature_importance = feature_importance.sort_values(
        "importance",
        ascending=False
    )

    display(feature_importance)
else:
    print("This model does not have feature_importances_.")

This model does not have feature_importances_.


In [18]:
# Coefficients for logistic regression
if hasattr(best_model, "coef_"):
    coefficients = pd.DataFrame({
        "feature": features,
        "coefficient": best_model.coef_[0]
    })

    coefficients = coefficients.sort_values(
        "coefficient",
        ascending=False
    )

    display(coefficients)
else:
    print("This model does not have coefficients.")

,feature,coefficient
7,last3_win_pct_diff,0.439613
3,win_pct_diff,0.049852
0,avg_points_scored_diff,0.029989
2,avg_point_diff_diff,0.025656
1,avg_points_allowed_diff,0.004333
5,last3_avg_points_allowed_diff,0.004112
4,last3_avg_points_scored_diff,0.001387
6,last3_avg_point_diff_diff,-0.002725


In [19]:
# Save model comparison results
model_results_df.to_csv(
    "../data/predictions/model_comparison_results.csv",
    index=False
)

results.to_csv(
    "../data/predictions/best_model_2025_predictions.csv",
    index=False
)

print("Saved model comparison results and best model predictions.")

Saved model comparison results and best model predictions.


In [20]:
# Final summary
print("Model comparison summary:")
display(model_results_df)

print("\nBest model:", best_model_name)
print("Best accuracy:", f"{best_accuracy:.2%}")
print("Previous best accuracy: 62.46%")

Model comparison summary:


,model,accuracy,accuracy_percent,difference_from_previous_best,difference_percent_points
0,Logistic Regression,0.624561,62.46,-0.000039,-0.00
1,Random Forest,0.607018,60.70,-0.017582,-1.76
2,Gradient Boosting,0.564912,56.49,-0.059688,-5.97



Best model: Logistic Regression
Best accuracy: 62.46%
Previous best accuracy: 62.46%


## Summary

In this notebook, I compared multiple model types using the same features and the same train/test setup.

The models compared were:

1. Logistic Regression
2. Random Forest
3. Gradient Boosting

The model trained on the 2018–2024 seasons and tested on the 2025 season. This keeps the evaluation realistic because the model uses past seasons to predict a later season.

The next step is to decide which model should become the current best model for the dashboard.